### Python Memory Management

**Memory Management is the process by which the Operating System (OS) allocates, tracks, protects, and deallocates memory for programs and processes to ensure efficient and safe execution**.

#### Objectives
* Allocate memory to programs.
* Deallocate memory when it is no longer needed.
* Maximize memory utilization.
* Prevent one process from accessing another process's memory.
* Support multitasking and efficient execution.


#### Types of Memory
* **Stack Memory**: Stores local variables and function calls. Managed automatically.

* **Heap Memory**: Stores dynamically allocated objects. Managed by the programmer or garbage collector.
    **Key Functions**
    - Allocation: Assigns memory to a process or variable.
    - Deallocation: Frees memory after use.
    - Protection: Prevents unauthorized memory access.

* **Virtual Memory**: Uses disk space to extend RAM, allowing larger programs to run.

* **Garbage Collection**: Automatically reclaims unused memory (e.g., Python, Java).

#### Memory Management Techniques

* **Contiguous Memory Allocation**
    - Paging
    - Segmentation
    - Virtual Memory
    - Swapping
    - Demand Paging

**In Python Memory is managed automatically by the Python Memory Manager.Objects are stored on the heap**.
**Reference counting and the Garbage Collector (GC) free unused memory automatically**.

#### Advantages
* Efficient use of RAM.
* Supports multiple programs simultaneously.
* Prevents memory leaks and corruption.
* Improves system performance and stability.

### Reference Counting

**Reference counting is CPython’s primary mechanism for automatic memory management. Every Python object contains a hidden counter (ob_refcnt) that tracks how many variables, data structures, or code blocks currently point to it**.

#### How Reference Counting WorksT

he life cycle of an object's reference counter follows four fundamental rules:

1. **Increments**: The count increases by one when an object is instantiated, assigned to a new variable name, appended into a container (like a list or dictionary), or passed as an argument to a function.

2. **Decrements**: The count decreases by one when a variable pointing to it is explicitly deleted using del, reassigned to a different value, or when its hosting function finishes execution and its local scope is destroyed.

3. **Immediate** Deallocation: As soon as an object’s reference count drops to exactly zero, Python instantly frees its allocated memory block

🔑 **Note**: To understand more about `GIL` and `ParallelProcessing` visit this notebook.

[Cuda_programming.ipynb](https://github.com/devendra-kashyap9/Ultimate-Python/blob/main/Notebooks/Cuda_programming.ipynb)

In [31]:
import sys

a = []
print(sys.getrefcount(a))  # one reference from 'a' abd one from getrefcount()

2


In [32]:
b = a
print(sys.getrefcount(b))

3


In [33]:
del b
print(sys.getrefcount(a))

2


### Garbage Collection

**Garbage Collection (GC)** is the automatic process of freeing memory occupied by objects that are no longer in use. It helps prevent memory leaks and manages memory efficiently.

---

#### Why is Garbage Collection Needed?

- Frees unused memory
- Prevents memory leaks
- Improves performance
- Reduces manual memory management

---

#### Generational Garbage Collection

Reference counting cannot remove **circular references**.

```python
a = []
b = []

a.append(b)
b.append(a)
```

Python's **Generational Garbage Collector** detects and removes these unreachable objects.

### Generations

- **Generation 0:** New objects
- **Generation 1:** Objects surviving Gen 0
- **Generation 2:** Long-lived objects

---

#### `gc` Module

Python provides the **`gc`** module to control the garbage collector.

```python
import gc
```

Common functions:

```python
gc.collect()      # Run garbage collection
gc.enable()       # Enable GC
gc.disable()      # Disable GC
gc.isenabled()    # Check status
```

---

#### Advantages

- Automatic memory management
- Prevents memory leaks
- Handles circular references
- Improves program stability

---

#### Disadvantages

- Small runtime overhead
- Occasional execution pauses

---

#### Summary

- Python automatically manages memory.
- **Reference Counting** removes most unused objects.
- **Generational Garbage Collection** removes circular references.
- The **`gc`** module provides manual control over garbage collection.

In [34]:
import gc

# enable garbage collection
gc.enable()

In [35]:
gc.isenabled()

True

In [36]:
gc.disable()
gc.isenabled()

False

In [37]:
gc.collect() # manually trigger the gc
            # number of unreachable objects is returned

0

In [38]:
# garbage collect stats
gc.get_stats()

[{'collections': 67, 'collected': 1552, 'uncollectable': 0},
 {'collections': 6, 'collected': 270, 'uncollectable': 0},
 {'collections': 7, 'collected': 51, 'uncollectable': 0}]

In [39]:
# get unreachable objects
gc.garbage

[]

### Memory Management Best Practices

1. **Use Local Variables:** Local variables have a shorter lifespan and are freed sooner than global variables.

2. **Avoid Circular References:** Circular references can lead to memory leaks if not properly managed.

3. **Use Generators:** Generators produce items one at a time and only keep one item in memory at a time, making them memory efficient.

4. **Explicitly Delete Objects:** Use the `del` statement to delete variables and objects explicitly.

5. **Profile Memory Usage:** Use memory profiling tools like `tracemalloc` and `memory_profiler` to identify memory leaks and optimize memory usage.

In [40]:
# handling circular reference
import gc

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")
        
    def __del__(self):
        print(f"Object {self.name} deleted")
        
# create circular reference
obj1 = MyObject('obj1')
obj2 = MyObject('obj2')
obj1.ref = obj2
obj2.ref = obj1


del obj1
del obj2


# manually trigger the garbage collection to delete objects, cuz they have circular reference
gc.collect()

Object obj1 created
Object obj2 created
Object obj1 deleted
Object obj2 deleted


9

In [41]:
# Generator for memory efficiency
def generate_number(n):
    for i in range(n):
        yield i
        
# using our generator function
for num in generate_number(10000):
    print(num)
    if num > 10: break

0
1
2
3
4
5
6
7
8
9
10
11


In [47]:
# profiling memory usage with `tracemalloc`

'''
tracemalloc is Python's built-in standard library module used to debug, monitor, and trace memory allocations. 
It allows you to find memory leaks, see exactly which line of code allocated the most memory, and compare snapshots 
of memory usage over time.
'''

import tracemalloc

def create_list():
    return [i for i in range(1000)]

def main():
    tracemalloc.start()
    
    create_list()
    
    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics("lineno")
    
    print("[ Top 10]")
    for stat in top_stats[:10]:
        print(stat)

In [46]:
main()

[ Top 10]
/Users/spy/Desktop/python/.venv/lib/python3.13/site-packages/jupyter_client/session.py:1008: size=8192 B, count=1, average=8192 B
/opt/anaconda3/lib/python3.13/json/decoder.py:361: size=4033 B, count=43, average=94 B
/Users/spy/Desktop/python/.venv/lib/python3.13/site-packages/IPython/core/compilerop.py:174: size=3089 B, count=35, average=88 B
/opt/anaconda3/lib/python3.13/codeop.py:118: size=2438 B, count=29, average=84 B
/Users/spy/Desktop/python/.venv/lib/python3.13/site-packages/traitlets/traitlets.py:750: size=1481 B, count=23, average=64 B
/Users/spy/Desktop/python/.venv/lib/python3.13/site-packages/traitlets/traitlets.py:1522: size=1440 B, count=12, average=120 B
/Users/spy/Desktop/python/.venv/lib/python3.13/site-packages/jupyter_client/session.py:103: size=1231 B, count=8, average=154 B
/Users/spy/Desktop/python/.venv/lib/python3.13/site-packages/IPython/core/compilerop.py:86: size=1218 B, count=17, average=72 B
/Users/spy/Desktop/python/.venv/lib/python3.13/site-pac